# 05 · Publication figures

Regenerates the Nature-style figures with the shared style module `lib/_figstyle.py`
(colourblind-safe Okabe–Ito palette, Helvetica-like fonts, editable vector PDF).

The full figure scripts live in the MsClaude package under `04_figures_main/` and
`05_figures_supplementary/`; here we show the density figure inline as an example.

In [ ]:
# --- notebook preamble: locate the reusable analysis library ------------------
import os, sys, warnings
warnings.filterwarnings("ignore")
LIB = os.path.join(os.path.dirname(os.path.abspath("__file__")), "lib")
if not os.path.isdir(LIB):
    LIB = os.path.join(os.getcwd(), "lib")
sys.path.insert(0, LIB)

# _lib_raman defines ROOT / DATA / MODEL_PATH / META_PATH (absolute paths to the
# original project). If you moved the data, override them here, e.g.:
#   import _lib_raman as L; L.ROOT = "/path/to/Project_Raman"; L.DATA = ...
import _lib_raman as L
print("Project ROOT :", L.ROOT)
print("Graph DATA   :", L.DATA)
print("Model        :", L.MODEL_PATH)
print("Classes      :", L.CLASSES)

import _figstyle as S
S.apply_style()
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

In [ ]:
# graph-density figure from the values computed in notebook 03
df = pd.read_csv('outputs/master_predictions_meta.csv')
from sklearn.metrics import roc_auc_score
val = df[df.split=='val']
rows=[]
for (gs,hop),g in val.groupby(['graph_set','hop']):
    aucs=[]
    for ci,c in enumerate(L.CLASSES):
        yb=(g.y_true==ci).astype(int)
        if yb.nunique()>1: aucs.append(roc_auc_score(yb,g[f'logit_{c}']))
    rows.append(dict(median_nodes=g.n_nodes.median(), mean_auc=np.mean(aucs)))
d=pd.DataFrame(rows).sort_values('median_nodes')
fig,ax=plt.subplots(figsize=S.figsize_mm(90,60))
ax.plot(d.median_nodes,d.mean_auc,'-o',color=S.OKABE_ITO[0],ms=3)
ax.set_xlabel('Median subgraph size (nodes)'); ax.set_ylabel('Mean one-vs-rest AUC')
ax.set_title('Graph-density optimisation'); S.save_fig(fig if False else fig, 'outputs', 'nb_density'); plt.show()

To regenerate **all** figures with source data + captions, run the packaged scripts:
```bash
python 04_figures_main/gen_M_levels.py
python 04_figures_main/gen_M_leakage.py
python 05_figures_supplementary/gen_S_density_saturation.py
python 05_figures_supplementary/gen_S_per_center.py
python 05_figures_supplementary/gen_S_loco.py
```